# DSM GA — Hyperparameter Tuning with Optuna

Uses Optuna to find optimal parameters for the DSM genetic algorithm.

## How to use
1. Set `PARAM_GROUP` in the configuration cell to one of:
   - `'weights'` — alpha, beta, gamma, delta
   - `'ga_operators'` — population_size, cxpb, mutpb, tournsize
   - `'clustering'` — target_clusters, max_clusters, consolidation_mode
   - `'fitness_and_clustering'` — weights + clustering
   - `'all'` — every parameter
2. Adjust `TRIAL_GENERATIONS`, `TRIAL_POPULATION`, `N_TRIALS` for speed vs. quality
3. Run all cells
4. The final cell runs a full GA with the best found parameters and exports to Excel

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import optuna
from dataclasses import replace as dc_replace
from dsm_ga import load_data, GAConfig, run_ga

optuna.logging.set_verbosity(optuna.logging.WARNING)

## Data

In [ ]:
freq_csv_path   = "consolidation_potential_DSM_scenario1_interaction_frequency.csv"
consol_csv_path = "consolidation_potential_DSM_scenario1_SA4_FTE2_2026-08-04-12h27.csv"

DSM_freq, DSM_consol, units, DSM_header_list = load_data(freq_csv_path, consol_csv_path)
print(f"Loaded {len(units)} units")

## Tuning configuration

In [ ]:
# ── Parameter group ─────────────────────────────────────────────────────────
# Choose ONE of:
#   'weights'               — alpha, beta, gamma, delta
#   'ga_operators'          — population_size, cxpb, mutpb, tournsize
#   'clustering'            — target_clusters, max_clusters, consolidation_mode
#   'fitness_and_clustering'— weights + clustering
#   'all'                   — every parameter
PARAM_GROUP = "weights"

# ── Trial budget ─────────────────────────────────────────────────────────────
TRIAL_GENERATIONS = 100   # reduced GA run per trial (full run uses config.n_generations)
TRIAL_POPULATION  = 100   # reduced population per trial
N_TRIALS          = 50    # number of Optuna trials
N_JOBS            = 1     # set to -1 for parallel trials

# ── Base config (frozen params stay at these values) ─────────────────────────
base_config = GAConfig()

## Objective function

In [ ]:
VALID_GROUPS = {"weights", "ga_operators", "clustering", "fitness_and_clustering", "all"}
assert PARAM_GROUP in VALID_GROUPS, f"PARAM_GROUP must be one of {VALID_GROUPS}"


def _suggest_weights(trial):
    """Sample 4 weights that sum to <= 1.0."""
    raw = [trial.suggest_float(f"raw_w{i}", 0.0, 1.0) for i in range(4)]
    total = sum(raw)
    if total > 1.0:
        raw = [w / total * 0.95 for w in raw]
    return raw[0], raw[1], raw[2], raw[3]


def build_trial_config(trial, param_group: str, base: GAConfig) -> GAConfig:
    """Return a GAConfig with trial-suggested values merged over base."""
    kwargs = {}

    if param_group in ("weights", "fitness_and_clustering", "all"):
        a, b, g, d = _suggest_weights(trial)
        kwargs["alpha"] = a
        kwargs["beta"]  = b
        kwargs["gamma"] = g
        kwargs["delta"] = d

    if param_group in ("clustering", "fitness_and_clustering", "all"):
        target = trial.suggest_int("target_clusters", 3, 15)
        kwargs["target_clusters"]    = target
        kwargs["max_clusters"]       = trial.suggest_int("max_clusters", target, 20)
        kwargs["consolidation_mode"] = trial.suggest_categorical(
            "consolidation_mode", ["once", "directional"]
        )

    if param_group in ("ga_operators", "all"):
        kwargs["population_size"] = trial.suggest_int("population_size", 50, 500, step=50)
        kwargs["cxpb"]            = trial.suggest_float("cxpb", 0.2, 0.9)
        kwargs["mutpb"]           = trial.suggest_float("mutpb", 0.01, 0.3)
        kwargs["tournsize"]        = trial.suggest_int("tournsize", 2, 10)

    return dc_replace(base, **kwargs)


def objective(trial):
    cfg = build_trial_config(trial, PARAM_GROUP, base_config)
    cfg = dc_replace(cfg, n_generations=TRIAL_GENERATIONS, population_size=TRIAL_POPULATION)
    _, _, fitness = run_ga(DSM_freq, DSM_consol, cfg, verbose=False)
    return fitness

## Run study

In [ ]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=N_TRIALS, n_jobs=N_JOBS, show_progress_bar=True)

print(f"\nBest trial: #{study.best_trial.number}")
print(f"Best fitness: {study.best_value:.4f}")
print("\nBest parameters:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

## Results visualisation

In [ ]:
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plt.sca(axes[0])
plot_optimization_history(study)
axes[0].set_title("Optimization history")
plt.sca(axes[1])
plot_param_importances(study)
axes[1].set_title("Parameter importances")
plt.tight_layout()
plt.show()

## Full run with best parameters

Applies the best Optuna parameters on top of `base_config` and runs a full-length GA, then exports to Excel.

In [ ]:
best_config = build_trial_config(study.best_trial, PARAM_GROUP, base_config)
print("Running full GA with best config:")
print(best_config)

best, logbook, final_fitness = run_ga(
    DSM_freq, DSM_consol, best_config, verbose=True
)
print(f"\nFinal fitness: {final_fitness:.4f}")
print(f"Best solution: {best}")

In [ ]:
from IPython.display import display

NUMBER_OF_DSM_ELEMENTS = len(units)
group_assignments = list(best)
sorted_indices = sorted(range(NUMBER_OF_DSM_ELEMENTS), key=lambda i: group_assignments[i])
sorted_units  = [DSM_header_list[i] for i in sorted_indices]
sorted_groups = [group_assignments[i] for i in sorted_indices]

df_optimized_grouping = pd.DataFrame({
    'Unit Name': sorted_units,
    'Cluster':   [f'Cluster {g}' for g in sorted_groups],
})
display(df_optimized_grouping)

csv_file = consol_csv_path.replace('.csv', '')
output_path = csv_file + f'_tuned_{PARAM_GROUP}_optimized.xlsx'

DSM_reordered = DSM_freq.iloc[sorted_indices].iloc[:, sorted_indices].copy()
DSM_reordered.index = DSM_reordered.columns = sorted_units
DSM_consol_reordered = DSM_consol.iloc[sorted_indices].iloc[:, sorted_indices].copy()
DSM_consol_reordered.index = DSM_consol_reordered.columns = sorted_units

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    DSM_reordered.to_excel(writer, sheet_name='dsm_optimized')
    DSM_consol_reordered.to_excel(writer, sheet_name='dsm_consolidation')
    df_optimized_grouping.to_excel(writer, sheet_name='grouping')
    DSM_freq.to_excel(writer, sheet_name='interaction_frequency')
    DSM_consol.to_excel(writer, sheet_name='consolidation_potential')

print(f'Exported to {output_path}')